In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import os

# Add the parent directory to Python's path
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import os
import src.model_architectures
from src.model_architectures import run_power_lstm, run_vanilla_transformer, run_univariate_patchtst, run_keras_patchtst, run_chronos_bolt, run_chronos2_multivariate, run_chronos_foundation
from src.ml_baselines import xgboost_implementation, rf_implementation
from src.evaluation_utils import save_experiment_results, plot_predictions, calculate_metrics
import importlib
import src.evaluation_utils
importlib.reload(src.evaluation_utils)
importlib.reload(src.model_architectures)
import importlib
import matplotlib.pyplot as plt
import random
import torch
import tensorflow as tf
import warnings
import logging

In [ ]:
#######################  30 - Minute Horizon Baselines #######################

# GLOBAL CONFIGURATION
HORIZON = "30min"
RANDOM_SEED = 42
horizon_metrics = []

# =======================================================
# NEW: 30-MINUTE MATHEMATICAL CONSTANTS (48 steps/day)
# =======================================================
SEQ_LENGTH = 48       # 24-hour sequence length for LSTM/Transformers
MASE_M = 48           # Seasonal denominator for MASE metric
PLOT_WINDOW = 336     # 7-day visual window for plots (48 * 7)

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import torch

# UNIVERSAL SEED LOCK
def set_global_seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    print(f"Global seed locked to {seed} for XGB, RF, LSTM, and Transformers.")

set_global_seed(RANDOM_SEED)

# LOAD & SPLIT DATA
# CRITICAL FIX: Ensure this path points to your 30-minute dataset!
df = pd.read_csv('../data/processed/nanogrid_30min_features.csv', index_col=0, parse_dates=True)
TARGET_COL = 'energy_consumption'

X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

# Deterministic Split (70/10/20)
total_rows = len(df)
train_idx = int(total_rows * 0.70)
val_idx = int(total_rows * 0.80)

X_train, y_train = X.iloc[:train_idx], y.iloc[:train_idx]
X_val, y_val     = X.iloc[train_idx:val_idx], y.iloc[train_idx:val_idx]
X_test, y_test   = X.iloc[val_idx:], y.iloc[val_idx:]

print(f"Features: {X.shape[1]} | Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

In [ ]:
# ==============================================================================
# CLASSICAL ML: XGBOOST & RANDOM FOREST (30-MIN)
# ==============================================================================
print("\nTraining Classical ML: XGBoost...")
xgb_results_30m = xgboost_implementation(
    X_train, y_train, 
    X_val, y_val, 
    X_test, y_test, 
    horizon_name=HORIZON
)
save_experiment_results(xgb_results_30m, "XGBoost", HORIZON, y_test)

# CRITICAL FIX: Recalculate metrics here to override the m=96 default!
xgb_metrics_30m = calculate_metrics(
    y_true=y_test,
    y_pred=xgb_results_30m["predictions"],
    model_name="XGBoost",
    execution_time=xgb_results_30m["metrics"]["Time (s)"], # Rescue the execution time
    m=MASE_M,                                              # Force m=48
    y_train=y_train                                        # Pass y_train for accurate scaling
)
horizon_metrics.append(xgb_metrics_30m)
plot_predictions(y_test, xgb_results_30m["predictions"], "XGBoost", HORIZON, num_steps=PLOT_WINDOW)


print("\nTraining Classical ML: Random Forest...")
rf_results_30m = rf_implementation(
    X_train, y_train, 
    X_val, y_val, 
    X_test, y_test, 
    horizon_name=HORIZON
)
save_experiment_results(rf_results_30m, "Random_Forest", HORIZON, y_test)

# CRITICAL FIX: Recalculate metrics here to override the m=96 default!
rf_metrics_30m = calculate_metrics(
    y_true=y_test,
    y_pred=rf_results_30m["predictions"],
    model_name="Random_Forest",
    execution_time=rf_results_30m["metrics"]["Time (s)"], 
    m=MASE_M, 
    y_train=y_train
)
horizon_metrics.append(rf_metrics_30m)
plot_predictions(y_test, rf_results_30m["predictions"], "Random_Forest", HORIZON, num_steps=PLOT_WINDOW)

In [ ]:
# ==============================================================================
# DEEP LEARNING: LSTM EXPERT (30-MINUTE HORIZON)
# ==============================================================================
print("\nTraining Deep Learning: Power-LSTM...")
lstm_results_30m = run_power_lstm(
    df=df, 
    target_col=TARGET_COL, 
    seq_length=SEQ_LENGTH,  # <--- Automatically 48!
    epochs=50, 
    batch_size=32
)

# 1. Save to Disk
save_experiment_results(lstm_results_30m, "LSTM", HORIZON, lstm_results_30m['y_test_real'])

# 2. Calculate Metrics
lstm_metrics_30m = calculate_metrics(
    y_true=lstm_results_30m['y_test_real'],
    y_pred=lstm_results_30m['predictions'],
    model_name="LSTM",
    execution_time=lstm_results_30m['execution_time'],
    m=MASE_M,               # <--- Automatically overrides to 48!
    y_train=lstm_results_30m['y_train_real']
)
horizon_metrics.append(lstm_metrics_30m)

# 3. Plot 7-Day Window
plot_predictions(
    y_true=lstm_results_30m['y_test_real'], 
    y_pred=lstm_results_30m['predictions'], 
    model_name="LSTM", 
    horizon=HORIZON, 
    num_steps=PLOT_WINDOW   # <--- Automatically 336 steps!
)

In [ ]:
# ==============================================================================
# DEEP LEARNING: VANILLA TRANSFORMER (30-MINUTE HORIZON)
# ==============================================================================
print("\nTraining Deep Learning: Vanilla Transformer...")
transformer_results_30m = run_vanilla_transformer(
    df=df, 
    target_col=TARGET_COL, 
    seq_length=SEQ_LENGTH,  # <--- Automatically 48!
    epochs=50, 
    batch_size=32 
)

# 1. Save to Disk
save_experiment_results(transformer_results_30m, "Vanilla_Transformer", HORIZON, transformer_results_30m['y_test_real'])

# 2. Calculate Metrics
transformer_metrics_30m = calculate_metrics(
    y_true=transformer_results_30m['y_test_real'],
    y_pred=transformer_results_30m['predictions'],
    model_name="Vanilla_Transformer",
    execution_time=transformer_results_30m['execution_time'],
    m=MASE_M,               # <--- Automatically overrides to 48!
    y_train=transformer_results_30m['y_train_real']
)
horizon_metrics.append(transformer_metrics_30m)

# 3. Plot 7-Day Window
plot_predictions(
    y_true=transformer_results_30m['y_test_real'], 
    y_pred=transformer_results_30m['predictions'], 
    model_name="Vanilla_Transformer", 
    horizon=HORIZON, 
    num_steps=PLOT_WINDOW   # <--- Automatically 336 steps!
)

In [ ]:
# ==============================================================================
# FOUNDATION ML: RESCUED PATCH-TST (30-MINUTE HORIZON)
# ==============================================================================
print("\nTraining Foundation ML: PatchTST (RevIN)...")
patch_results_30m = run_keras_patchtst(
    df=df, 
    target_col=TARGET_COL, 
    seq_length=SEQ_LENGTH,  # <--- Automatically 48!
    epochs=50, 
    batch_size=32,
    patch_len=16, 
    stride=8
)

# 1. Save to Disk
save_experiment_results(patch_results_30m, "PatchTST_RevIN", HORIZON, patch_results_30m['y_test_real'])

# 2. Calculate Metrics
patch_metrics_30m = calculate_metrics(
    y_true=patch_results_30m['y_test_real'],
    y_pred=patch_results_30m['predictions'],
    model_name="PatchTST_RevIN",
    execution_time=patch_results_30m['execution_time'],
    m=MASE_M,               # <--- Automatically overrides to 48!
    y_train=patch_results_30m['y_train_real']
)
horizon_metrics.append(patch_metrics_30m)

# 3. Plot 7-Day Window
plot_predictions(
    y_true=patch_results_30m['y_test_real'], 
    y_pred=patch_results_30m['predictions'], 
    model_name="PatchTST_RevIN", 
    horizon=HORIZON, 
    num_steps=PLOT_WINDOW   # <--- Automatically 336 steps!
)

In [ ]:
# ==============================================================================
# FOUNDATION ML: CHRONOS-BOLT (UNIVARIATE BASELINE)
# ==============================================================================
print("\nTraining Foundation ML: Chronos-Bolt (Small)...")
chronos_results_30m = run_chronos_foundation(
    df=df, 
    target_col=TARGET_COL, 
    seq_length=SEQ_LENGTH,   # Automatically 48!
    batch_size=16,           # Bolt can handle larger batches!
    model_name="amazon/chronos-bolt-small" # Using the vastly superior Bolt architecture
)

# 1. Save to Disk
save_experiment_results(chronos_results_30m, "Chronos_Bolt", HORIZON, chronos_results_30m['y_test_real'])

# 2. Calculate Metrics
chronos_metrics_30m = calculate_metrics(
    y_true=chronos_results_30m['y_test_real'],
    y_pred=chronos_results_30m['predictions'],
    model_name="Chronos_Bolt",
    execution_time=chronos_results_30m['execution_time'],
    m=MASE_M,                # Automatically 48!
    y_train=chronos_results_30m['y_train_real']
)
horizon_metrics.append(chronos_metrics_30m)

# 3. Plot 7-Day Window
plot_predictions(
    y_true=chronos_results_30m['y_test_real'], 
    y_pred=chronos_results_30m['predictions'], 
    model_name="Chronos_Bolt", 
    horizon=HORIZON, 
    num_steps=PLOT_WINDOW    # Automatically 336 steps!
)

In [ ]:
# ==============================================================================
# FOUNDATION ML: CHRONOS-2 MULTIVARIATE (THE "FAIR FIGHT")
# ==============================================================================
print("\nTraining Foundation ML: Chronos-2 (Multivariate with Covariates)...")
chronos2_results_30m = run_chronos2_multivariate(
    df=df, 
    target_col=TARGET_COL, 
    prediction_length=SEQ_LENGTH, # <--- Automatically 48!
    model_name="amazon/chronos-2", # Or whichever specific v2 string you have downloaded
    batch_size=8
)

# 1. Save to Disk
save_experiment_results(chronos2_results_30m, "Chronos_2_Multivariate", HORIZON, chronos2_results_30m['y_test_real'])

# 2. Calculate Metrics
chronos2_metrics_30m = calculate_metrics(
    y_true=chronos2_results_30m['y_test_real'], 
    y_pred=chronos2_results_30m['predictions'],
    model_name="Chronos_2_Multivariate", 
    execution_time=chronos2_results_30m['execution_time'], 
    m=MASE_M,                     # <--- Automatically overrides to 48!
    y_train=chronos2_results_30m['y_train_real']
)
horizon_metrics.append(chronos2_metrics_30m)

# 3. Plot 7-Day Window
plot_predictions(
    y_true=chronos2_results_30m['y_test_real'], 
    y_pred=chronos2_results_30m['predictions'], 
    model_name="Chronos_2_Multivariate", 
    horizon=HORIZON, 
    num_steps=PLOT_WINDOW         # <--- Automatically 336 steps!
)